In [1]:
import yfinance as yf
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score

In [2]:
# Download S&P 500 index historical data
sp500 = yf.Ticker("^GSPC")
sp500 = sp500.history(period="max")

# Display the initial data structure
print("Initial data shape:", sp500.shape)
display(sp500.head())

# Clean unnecessary columns
sp500 = sp500.drop(columns=["Dividends", "Stock Splits"])

# Setup target: Predict if tomorrow's price is higher than today's (1 for Up, 0 for Down)
sp500["Tomorrow"] = sp500["Close"].shift(-1)
sp500["Target"] = (sp500["Tomorrow"] > sp500["Close"]).astype(int)

# Remove data prior to 1990 to focus on modern market behavior
sp500 = sp500.loc["1990-01-01":].copy()

Initial data shape: (24811, 7)


,Open,High,Low,Close,Volume,Dividends,Stock Splits
Date,,,,,,,
1927-12-30 00:00:00-05:00,17.660000,17.660000,17.660000,17.660000,0,0.0,0.0
1928-01-03 00:00:00-05:00,17.760000,17.760000,17.760000,17.760000,0,0.0,0.0
1928-01-04 00:00:00-05:00,17.719999,17.719999,17.719999,17.719999,0,0.0,0.0
1928-01-05 00:00:00-05:00,17.549999,17.549999,17.549999,17.549999,0,0.0,0.0
1928-01-06 00:00:00-05:00,17.660000,17.660000,17.660000,17.660000,0,0.0,0.0


In [4]:
# Create rolling window features to capture historical trends over different time horizons
horizons = [2, 5, 60, 250, 1000]
new_predictors = []

for horizon in horizons:
    rolling_average = sp500["Close"].rolling(horizon).mean()

    ratio_column = f"Close_Ratio_{horizon}"
    sp500[ratio_column] = sp500["Close"] / rolling_average

    trend_column = f"Trend_{horizon}"
    sp500[trend_column] = sp500["Target"].shift(1).rolling(horizon).sum()

    new_predictors += [ratio_column, trend_column]

# Drop missing values resulting from rolling windows
sp500 = sp500.dropna()

In [5]:
# Initialize a Random Forest Classifier model
model = RandomForestClassifier(n_estimators=200, min_samples_split=50, random_state=1)

def predict(train, test, predictors, model):
    model.fit(train[predictors], train["Target"])
    preds = model.predict(test[predictors])
    preds = pd.Series(preds, index=test.index, name="Predictions")
    combined = pd.concat([test["Target"], preds], axis=1)
    return combined

def backtest(data, model, predictors, start=2500, step=250):
    all_predictions = []

    for i in range(start, data.shape[0], step):
        train = data.iloc[0:i].copy()
        test = data.iloc[i:(i+step)].copy()
        predictions = predict(train, test, predictors, model)
        all_predictions.append(predictions)

    return pd.concat(all_predictions)

# Execute backtest using engineered features
predictions = backtest(sp500, model, new_predictors)

# Evaluate precision score
precision = precision_score(predictions["Target"], predictions["Predictions"])
print(f"Model Precision Score: {precision * 100:.2f}%")

Model Precision Score: 55.27%
